# arXiv Computer Science 최근 1개월 논문 PDF 텍스트 수집

arXiv API에서 Computer Science(`cs.*`) 카테고리이면서 최근 1개월 이내에 제출된 논문을 조회하고, 각 논문의 PDF를 내려받아 텍스트를 추출한 뒤 JSONL로 저장합니다.

> PDF는 `E:\pdf`에 캐시합니다. 기본 수집 상한은 100건이며, PDF 다운로드에는 시간이 걸릴 수 있습니다.

In [ ]:
# 프로젝트 환경에 pypdf가 없다면 별도 셀에서 다음을 실행하세요.
# %pip install pypdf

from calendar import monthrange
from datetime import date
from io import BytesIO
from pathlib import Path
import hashlib
import json
import re
import ssl
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

try:
    import pypdf
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        'pypdf가 필요합니다. 위의 %pip install pypdf 셀을 실행한 뒤 다시 시도하세요.'
    ) from error

MAX_RESULTS = 100
PAGE_SIZE = 50
REQUEST_INTERVAL_SECONDS = 5.0
MAX_RETRIES = 6
BACKOFF_BASE_SECONDS = 5.0
BACKOFF_MAX_SECONDS = 120.0
API_URL = 'https://export.arxiv.org/api/query'
USER_AGENT = 'arxiv-cs-pdf-text-jsonl/1.0'

def subtract_months(day, months):
    month_index = day.year * 12 + day.month - 1 - months
    year, month_zero_based = divmod(month_index, 12)
    month = month_zero_based + 1
    return date(year, month, min(day.day, monthrange(year, month)[1]))

END_DATE = date.today()
START_DATE = subtract_months(END_DATE, 1)
DATE_QUERY = f'submittedDate:[{START_DATE:%Y%m%d}0000 TO {END_DATE:%Y%m%d}2359]'
SEARCH_QUERY = f'cat:cs.* AND {DATE_QUERY}'

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
PDF_CACHE_DIR = Path(r'E:\pdf')
OUTPUT_PATH = DATA_DIR / 'arxiv_cs_recent_1month_pdf_text.jsonl'
PDF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'조회 기간: {START_DATE} ~ {END_DATE}')
print(f'검색식: {SEARCH_QUERY}')
print(f'PDF 캐시: {PDF_CACHE_DIR.resolve()}')
print(f'저장 위치: {OUTPUT_PATH.resolve()}')

In [ ]:
ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom'}
ARXIV_NS = {'arxiv': 'http://arxiv.org/schemas/atom'}

def text_or_none(parent, path, namespaces=ATOM_NS):
    node = parent.find(path, namespaces)
    return node.text.strip() if node is not None and node.text else None

def parse_entry(entry):
    authors = [
        name
        for author in entry.findall('atom:author', ATOM_NS)
        for name in [text_or_none(author, 'atom:name')]
        if name
    ]
    categories = [
        node.attrib['term']
        for node in entry.findall('atom:category', ATOM_NS)
        if 'term' in node.attrib
    ]
    links = {
        link.attrib.get('rel', 'alternate'): link.attrib.get('href')
        for link in entry.findall('atom:link', ATOM_NS)
    }
    primary_node = entry.find('arxiv:primary_category', ARXIV_NS)
    return {
        'id': text_or_none(entry, 'atom:id'),
        'title': ' '.join((text_or_none(entry, 'atom:title') or '').split()),
        'abstract': ' '.join((text_or_none(entry, 'atom:summary') or '').split()),
        'authors': authors,
        'categories': categories,
        'primary_category': primary_node.attrib.get('term') if primary_node is not None else None,
        'published': text_or_none(entry, 'atom:published'),
        'updated': text_or_none(entry, 'atom:updated'),
        'doi': text_or_none(entry, 'arxiv:doi', ARXIV_NS),
        'pdf_url': links.get('related') or links.get('alternate'),
        'source': 'arxiv',
        'collection_window': {'start': START_DATE.isoformat(), 'end': END_DATE.isoformat()},
    }

def _retry_delay(error, attempt):
    retry_after = error.headers.get('Retry-After') if getattr(error, 'headers', None) else None
    if retry_after:
        try:
            return max(float(retry_after), REQUEST_INTERVAL_SECONDS)
        except ValueError:
            pass
    return min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)

_last_request_at = 0.0

def _urlopen_with_retry(request, timeout):
    global _last_request_at
    for attempt in range(MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        try:
            _last_request_at = time.monotonic()
            return urllib.request.urlopen(request, timeout=timeout, context=ssl.create_default_context())
        except urllib.error.HTTPError as error:
            if error.code not in {429, 500, 502, 503, 504} or attempt >= MAX_RETRIES:
                raise
            delay = _retry_delay(error, attempt)
            print(f'HTTP {error.code}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            error.close()
            time.sleep(delay)

def fetch_page(start=0, max_results=PAGE_SIZE):
    params = {
        'search_query': SEARCH_QUERY,
        'start': start,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending',
    }
    url = f'{API_URL}?{urllib.parse.urlencode(params)}'
    request = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
    with _urlopen_with_retry(request, timeout=60) as response:
        root = ET.fromstring(response.read())
    return [parse_entry(entry) for entry in root.findall('atom:entry', ATOM_NS)]

def cache_path_for(paper_id):
    key = hashlib.sha256(paper_id.encode('utf-8')).hexdigest()[:20]
    return PDF_CACHE_DIR / f'{key}.pdf'

def download_pdf(pdf_url, destination):
    if destination.exists() and destination.stat().st_size > 0:
        return 'cached'
    request = urllib.request.Request(pdf_url, headers={'User-Agent': USER_AGENT})
    with _urlopen_with_retry(request, timeout=120) as response:
        content = response.read()
    if not content.startswith(b'%PDF'):
        raise ValueError('응답이 PDF 파일이 아닙니다.')
    destination.write_bytes(content)
    return 'downloaded'

def extract_pdf_text(pdf_path):
    reader = pypdf.PdfReader(str(pdf_path))
    pages = [(page.extract_text() or '') for page in reader.pages]
    text = '\n\n'.join(page.strip() for page in pages if page.strip())
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip(), len(reader.pages)

In [ ]:
papers = []
seen_ids = set()
for start in range(0, MAX_RESULTS, PAGE_SIZE):
    page = fetch_page(start=start, max_results=min(PAGE_SIZE, MAX_RESULTS - start))
    for paper in page:
        if paper['id'] and paper['id'] not in seen_ids:
            seen_ids.add(paper['id'])
            papers.append(paper)
    print(f'{len(papers)}개 메타데이터 수집 완료')
    if len(papers) >= MAX_RESULTS or len(page) < PAGE_SIZE:
        break
    time.sleep(REQUEST_INTERVAL_SECONDS)

papers = papers[:MAX_RESULTS]
records = []
for index, paper in enumerate(papers, start=1):
    record = dict(paper)
    try:
        pdf_path = cache_path_for(paper['id'])
        record['pdf_cache_path'] = str(pdf_path)
        record['pdf_status'] = download_pdf(paper['pdf_url'], pdf_path)
        record['text'], record['page_count'] = extract_pdf_text(pdf_path)
        record['text_length'] = len(record['text'])
        record['error'] = None
    except Exception as error:
        record['pdf_status'] = 'error'
        record['text'] = ''
        record['text_length'] = 0
        record['page_count'] = None
        record['error'] = f'{type(error).__name__}: {error}'
    records.append(record)
    with OUTPUT_PATH.open('w', encoding='utf-8') as file:
        for saved_record in records:
            file.write(json.dumps(saved_record, ensure_ascii=False) + '\n')
    print(f'[{index}/{len(papers)}] {record["pdf_status"]}: {record["title"][:70]}')
    if index < len(papers):
        time.sleep(REQUEST_INTERVAL_SECONDS)

with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    for record in records:
        file.write(json.dumps(record, ensure_ascii=False) + '\n')

success_count = sum(record['error'] is None for record in records)
print(f'완료: {len(records)}개 레코드 저장, PDF 처리 성공 {success_count}개')
print(f'저장 위치: {OUTPUT_PATH.resolve()}')

In [ ]:
# JSONL 저장 결과 확인
with OUTPUT_PATH.open(encoding='utf-8') as file:
    saved_records = [json.loads(line) for line in file if line.strip()]

assert len(saved_records) == len(records)
assert len({record['id'] for record in saved_records}) == len(saved_records)
assert all(record['id'] and record['title'] for record in saved_records)
assert all(START_DATE.isoformat() <= record['published'][:10] <= END_DATE.isoformat() for record in saved_records)
assert all('text' in record and 'error' in record for record in saved_records)
print(f'검증 완료: {len(saved_records)}개 레코드, 중복 없음, 최근 1개월 범위 확인')
display(saved_records[:2])